In [2]:
%cd ../

/home/syed/VS code/globalretail-360


## Hypothesis 1: The Discount Trap

### Business Question
Do high discounts (>20%) increase the likelihood of product returns compared to low discounts (<10%)?

This analysis evaluates whether aggressive discounting attracts lower-quality transactions that result in higher return rates.

---

## Why This Matters

Discounting is commonly used to drive sales volume.  
However, if high discounts significantly increase return rates, the company may be sacrificing long-term profitability for short-term revenue.

Understanding this relationship informs:

- Pricing strategy  
- Promotion policy  
- Margin protection  
- Customer targeting decisions  

---

## Statistical Framing

We are testing whether **return rate is independent of discount level**.

### Independent Variable (IV)
Discount Category:
- Low Discount (<10%)
- High Discount (>20%)

### Dependent Variable (DV)
Return Flag:
- Returned (1)
- Not Returned (0)

This is a test of **difference in proportions**.

---

## Formal Hypotheses

Let:

- \( p_H \) = Return rate for High Discount orders (>20%)
- \( p_L \) = Return rate for Low Discount orders (<10%)

### Null Hypothesis (H₀)

The return rate is independent of discount level.

\[
H_0: p_H = p_L
\]

There is no statistically significant difference in return rates between high and low discount orders.

---

### Alternative Hypothesis (H₁)

High-discount orders have a higher return rate than low-discount orders.

\[
H_1: p_H > p_L
\]

This is a **one-tailed hypothesis**, because the business concern is specifically whether aggressive discounting increases returns.

In [3]:
import pandas as pd
from sqlalchemy import create_engine
from src.utils.config import ENV, POSTGRES_URI

engine = create_engine(POSTGRES_URI)

query = """
SELECT
    discount,
    return_flag
FROM fact_sales
WHERE discount IS NOT NULL;
"""

df = pd.read_sql(query, engine)

df.head()

,discount,return_flag
0,0.45,0
1,0.20,0
2,0.40,0
3,0.00,0
4,0.00,0


In [4]:
df.shape

(51593, 2)

In [5]:
low_discount = df[df["discount"] < 0.10].copy()
high_discount = df[df["discount"] > 0.20].copy()

low_discount["discount_group"] = "Low (<10%)"
high_discount["discount_group"] = "High (>20%)"

df_filtered = pd.concat([low_discount, high_discount])

df_filtered.head()

,discount,return_flag,discount_group
3,0.0,0,Low (<10%)
4,0.0,0,Low (<10%)
5,0.0,0,Low (<10%)
7,0.0,0,Low (<10%)
10,0.0,0,Low (<10%)


In [6]:
df_filtered["discount_group"].value_counts()

discount_group
Low (<10%)     29807
High (>20%)    11398
Name: count, dtype: int64

---

## Data Filtering & Group Definition

To create a clear comparison between discount extremes, we excluded orders with discounts between 10% and 20%.

### Rationale

The mid-range (10%–20%) represents moderate discounting and may blur the contrast between aggressive and minimal pricing strategies.

By focusing on:

- **Low Discount (<10%)**
- **High Discount (>20%)**

we create a sharper comparison, improving interpretability and avoiding dilution of the statistical signal.

### Final Sample Sizes

- Low Discount: 29,807 orders  
- High Discount: 11,398 orders  

Both groups have sufficiently large sample sizes for reliable inference.

In [7]:
return_rates = (
    df_filtered
    .groupby("discount_group")["return_flag"]
    .mean()
)

return_counts = (
    df_filtered
    .groupby("discount_group")["return_flag"]
    .agg(["sum", "count"])
)

return_rates, return_counts

(discount_group
 High (>20%)    0.133444
 Low (<10%)     0.112826
 Name: return_flag, dtype: float64,
                  sum  count
 discount_group             
 High (>20%)     1521  11398
 Low (<10%)      3363  29807)

## Descriptive Results (Before Statistical Testing)

### Return Rates

- **High Discount (>20%)**: 13.34%
- **Low Discount (<10%)**: 11.28%

### Absolute Difference

High-discount orders exhibit a **2.06 percentage point higher return rate** compared to low-discount orders.

### Preliminary Interpretation

At a descriptive level, aggressive discounting appears associated with a higher return rate.

However, descriptive differences alone do not indicate statistical significance.  
We must formally test whether this difference is unlikely to have occurred by random variation.

In [8]:
contingency_table = pd.crosstab(
    df_filtered["discount_group"],
    df_filtered["return_flag"]
)

contingency_table

return_flag,0,1
discount_group,,
High (>20%),9877,1521
Low (<10%),26444,3363


In [9]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

chi2, p_value, dof

(np.float64(33.35216913592821), np.float64(7.68920085104003e-09), 1)

In [10]:
expected

array([[10047.00298507,  1350.99701493],
       [26273.99701493,  3533.00298507]])

In [11]:
import numpy as np

n = contingency_table.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency_table.shape) - 1)))

cramers_v

np.float64(0.028450315248027172)

---

## Chi-Square Test Results

- **Chi-square statistic:** 33.35  
- **Degrees of freedom:** 1  
- **p-value:** < 0.001  
- **Cramér’s V:** 0.028  

### Statistical Decision

Because the p-value is far below 0.05, we reject the null hypothesis.

There is a statistically significant association between discount level and return rate.

---

## Effect Size Interpretation

Although statistically significant, the effect size (Cramér’s V = 0.028) is extremely small.

This indicates that:

- Discount level explains only a very small portion of variation in return behavior.
- The relationship, while real, is weak in magnitude.

---

## Business Interpretation

High-discount orders (>20%) show a 2.06 percentage point higher return rate than low-discount orders (<10%).

However, the practical impact of this difference is limited.

This suggests that:

- Aggressive discounting slightly increases return risk,
- But discounting alone is unlikely to be a major driver of returns.
- Other operational or customer-related factors may play a larger role.

---

## Key Insight

Large datasets can produce statistically significant results even when practical impact is minimal.

For decision-making, both **statistical significance** and **effect size** must be considered.

In [12]:
# Return rates
p_high = return_rates["High (>20%)"]
p_low = return_rates["Low (<10%)"]

# Sample sizes
n_high = return_counts.loc["High (>20%)", "count"]
n_low = return_counts.loc["Low (<10%)", "count"]

# Difference in proportions
diff = p_high - p_low

# Standard error
se = np.sqrt( (p_high*(1-p_high)/n_high) + (p_low*(1-p_low)/n_low) )

# 95% CI
from scipy.stats import norm
z = norm.ppf(0.975)  # 1.96
ci_lower = diff - z*se
ci_upper = diff + z*se

diff, ci_lower, ci_upper

(np.float64(0.02061861766332229),
 np.float64(0.013416310561491916),
 np.float64(0.027820924765152667))

---

## Confidence Interval for Difference in Return Rates

- Difference in return rates (High − Low): **2.06 percentage points**  
- 95% Confidence Interval: **1.34% – 2.78%**

### Interpretation

We are 95% confident that high-discount orders (>20%) have a return rate **between 1.34 and 2.78 percentage points higher** than low-discount orders (<10%).

### Business Insight

- The increase in return rate is **statistically significant**, but **practically small**.
- Aggressive discounting slightly increases returns, but the effect is minor in magnitude.
- Decision-makers should consider other operational factors when managing returns.
- Discount strategy may not be the main driver of returns, though small adjustments could optimize profitability.

---

### Key Takeaways

- Statistical significance ≠ practical significance  
- Confidence intervals communicate uncertainty in a business-friendly way  
- Effect sizes (Cramér’s V) contextualize the magnitude of the effect  

This completes a **full, professional analysis** of Hypothesis 1.

## Hypothesis 2: Shipping Leakage

### Business Question
Do “Critical” priority orders shipped via First Class in the Consumer segment frequently result in negative net profit margins?

This analysis evaluates whether high-cost shipping on high-priority orders is causing significant profitability loss for the company.

---

## Why This Matters

- High shipping costs can erode margins, especially on “Critical” priority orders.
- Identifying these inefficiencies helps:
  - Reduce losses
  - Improve unit economics
  - Optimize fulfillment and shipping strategy
- Specifically, we are testing whether ≥40% of such orders yield negative net profit margins.

---

## Statistical Framing

- **Independent Variables (IVs):**
  - Order Priority = “Critical”
  - Shipping Mode = “First Class”
  - Customer Segment = “Consumer”

- **Dependent Variable (DV):**
  - Net Profit Margin per order
    - Negative Margin (1) vs Non-Negative Margin (0)

We are testing **proportion of orders with negative margins**.

In [13]:
engine = create_engine(POSTGRES_URI)

query = """
SELECT
    f.order_id,
    c.segment,
    o."Order Priority" as order_priority,
    o."Ship Mode" as ship_mode,
    f.profit,
    f.sales
FROM fact_sales f
JOIN dim_customers c
    ON f.customer_key = c.customer_key
JOIN orders o
    ON f.order_id = o."Order ID"
WHERE c.segment IS NOT NULL
  AND o."Order Priority" IS NOT NULL
  AND o."Ship Mode" IS NOT NULL
  AND f.profit IS NOT NULL
  AND f.sales IS NOT NULL;
"""

df_sales = pd.read_sql(query, engine)

df_sales['net_margin'] = df_sales['profit'] / df_sales['sales']

df_filtered = df_sales[
    (df_sales['segment'] == 'Consumer') &
    (df_sales['order_priority'] == 'Critical') &
    (df_sales['ship_mode'] == 'First Class')
].copy()

leakage_proportion = (df_filtered['net_margin'] < 0).mean()
print(f"Proportion of Critical Consumer First Class orders with negative net margin: {leakage_proportion:.2%}")

Proportion of Critical Consumer First Class orders with negative net margin: 25.94%


**Result:**  
- **Proportion of negative net margin orders:** 25.94%  
- **Interpretation:** The hypothesis is **not supported**. Fewer than 40% of these high-priority orders are losing money, though a significant minority (~26%) still has negative margins.

## Hypothesis 3: Segment Value (Customer Modeling / Churn)

### Business Question
Does the Corporate segment have significantly higher retention and lower return rates than the Consumer segment over a 12-month period?

This analysis evaluates whether segment-level customer lifetime value (LTV) assumptions are valid, informing marketing spend and acquisition strategy.

---

## Why This Matters

- Understanding retention differences by segment helps optimize marketing and loyalty investments.
- Lower return rates in Corporate customers indicate **more profitable segment behavior**.
- Validates assumptions about LTV differences between Consumer and Corporate customers.

---

## Statistical Framing

- **Independent Variable (IV):** Customer Segment (Corporate vs Consumer)  
- **Dependent Variables (DVs):**  
  - 12-month retention (binary: retained vs churned)  
  - Return rate (percentage of orders returned)  

We are testing **whether Corporate customers have ≥20% higher retention and lower return rate than Consumer customers**.

In [23]:
engine = create_engine(POSTGRES_URI)

query = """
SELECT
    c.customer_id,
    c.segment,
    o."Order Date" AS order_date,
    f.return_flag
FROM dim_customers c
JOIN orders o
    ON c.customer_id = o."Customer ID"
JOIN fact_sales f
    ON o."Order ID" = f.order_id
WHERE c.segment IN ('Corporate', 'Consumer')
  AND o."Order Date" IS NOT NULL;
"""

df_customers = pd.read_sql(query, engine)

df_customers['order_date'] = pd.to_datetime(df_customers['order_date'], errors='coerce')

missing_dates = df_customers['order_date'].isna().sum()
print(f"Missing order dates: {missing_dates}")

retention_df = (
    df_customers.groupby(['segment', 'customer_id'])
    .agg(first_order=('order_date', 'min'),
         last_order=('order_date', 'max'))
    .reset_index()
)

snapshot_date = df_customers['order_date'].max()
print(f"Snapshot date: {snapshot_date}")

retention_df['retained_12mo'] = (snapshot_date - retention_df['last_order']).dt.days <= 365

segment_retention = (
    retention_df.groupby('segment')['retained_12mo']
    .mean()
    .reset_index()
    .rename(columns={'retained_12mo': 'retention_rate'})
)

print(segment_retention)

Missing order dates: 0
Snapshot date: 2017-12-31 00:00:00
     segment  retention_rate
0   Consumer        0.436965
1  Corporate        0.435740


In [ ]:
returns_df = (
    df_customers[df_customers['return_flag'] == 1]
    .groupby(['segment', 'customer_id'])
    .size()
    .reset_index(name='num_returns')
)

retention_with_returns = retention_df.merge(returns_df, on=['segment', 'customer_id'], how='left')
retention_with_returns['num_returns'] = retention_with_returns['num_returns'].fillna(0)

retention_with_returns['has_return'] = retention_with_returns['num_returns'] > 0
segment_return_stats = (
    retention_with_returns.groupby('segment')['has_return']
    .mean()
    .reset_index()
    .rename(columns={'has_return': 'fraction_retained_with_return'})
)

print(segment_return_stats)

     segment  fraction_retained_with_return
0   Consumer                       0.108713
1  Corporate                       0.110898


### Observations:

#### Retention: Both segments have almost identical 12-month retention (~44%). Corporate customers do not exhibit higher retention than Consumer customers.

#### Returns: The proportion of retained customers who made at least one return is similar (~11%), suggesting no meaningful difference in return behavior between segments.

#### Implication for LTV assumptions: Initial assumptions that Corporate customers have higher retention or lower return rates are not supported by the data.